# Assignment 3 — TinyML Cosine Wave Predictor
**SWAPD 453 — IoT Applications Development | Spring 2026**

This notebook walks the full edge-AI pipeline:
1. Data generation
2. Model training & evaluation
3. Full-integer (int8) quantization
4. C-header generation (`model.h`)

Run every cell **top-to-bottom** in Google Colab.  
At the end, download `model.h`, `cosine_float.tflite`, and `cosine_int8.tflite`.

## Step 0 — Install / verify dependencies

In [ ]:
# Colab already has TensorFlow; this cell just confirms versions.
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
print('TensorFlow :', tf.__version__)
print('NumPy      :', np.__version__)

## Step 1 — Data Generation

Generate 1000 samples of x ∈ [0, 2π] with y = cos(x) + Gaussian noise (σ = 0.1).  
Split: **60 % train / 20 % validation / 20 % test** (fixed seed for reproducibility).

In [ ]:
SEED    = 42
SAMPLES = 1000
np.random.seed(SEED)

# --- raw data ---
x_all = np.random.uniform(0, 2 * np.pi, SAMPLES).astype(np.float32)
y_all = (np.cos(x_all) + 0.1 * np.random.randn(SAMPLES)).astype(np.float32)

# --- shuffle (already random, but explicit for clarity) ---
idx = np.random.permutation(SAMPLES)
x_all, y_all = x_all[idx], y_all[idx]

# --- split ---
TRAIN_END = int(0.6 * SAMPLES)   # 600
VAL_END   = int(0.8 * SAMPLES)   # 800

x_train, y_train = x_all[:TRAIN_END],        y_all[:TRAIN_END]
x_val,   y_val   = x_all[TRAIN_END:VAL_END], y_all[TRAIN_END:VAL_END]
x_test,  y_test  = x_all[VAL_END:],          y_all[VAL_END:]

print(f'Train : {len(x_train)} samples')
print(f'Val   : {len(x_val)}   samples')
print(f'Test  : {len(x_test)}  samples')

# --- quick plot ---
x_dense = np.linspace(0, 2 * np.pi, 500)
plt.figure(figsize=(9, 3))
plt.scatter(x_train, y_train, s=4, alpha=0.5, label='train')
plt.scatter(x_val,   y_val,   s=4, alpha=0.5, label='val')
plt.scatter(x_test,  y_test,  s=4, alpha=0.5, label='test')
plt.plot(x_dense, np.cos(x_dense), 'k--', linewidth=1.5, label='cos(x)')
plt.title('Dataset — cosine + noise')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.tight_layout()
plt.savefig('plot_dataset.png', dpi=150); plt.show()
print('Saved: plot_dataset.png')

## Step 2 — Model Definition & Training

Architecture (exact as required):  
`Dense(16, ReLU) → Dense(16, ReLU) → Dense(1, linear)`  
Optimizer: Adam | Loss: MSE | Epochs: 500 | Batch: 64

In [ ]:
from tensorflow import keras

tf.random.set_seed(SEED)

model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(1,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1)   # linear output
], name='cosine_predictor')

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()
print(f'Trainable parameters: {model.count_params()}')

In [ ]:
EPOCHS     = 500
BATCH_SIZE = 64

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)

history = model.fit(
    x_train, y_train,
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    validation_data = (x_val, y_val),
    callbacks       = [early_stop],
    verbose         = 1
)

### 2a — Training & Validation Loss Curves

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history.history['loss'],     label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training & Validation Loss (MSE)')
plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.legend(); plt.tight_layout()
plt.savefig('plot_loss_curves.png', dpi=150); plt.show()
print('Saved: plot_loss_curves.png')

## Step 3 — Float Model Evaluation

In [ ]:
test_mse, test_mae = model.evaluate(x_test, y_test, verbose=0)
print(f'Float model — Test MSE : {test_mse:.6f}')
print(f'Float model — Test MAE : {test_mae:.6f}')
assert test_mae < 0.10, f'MAE {test_mae:.4f} is too high — train longer or check data.'

In [ ]:
# Float predictions on a dense grid for plotting
x_dense   = np.linspace(0, 2 * np.pi, 500).astype(np.float32)
y_gt      = np.cos(x_dense)
y_float   = model.predict(x_dense, verbose=0).flatten()

plt.figure(figsize=(9, 4))
plt.plot(x_dense, y_gt,    'k--', linewidth=1.5, label='Ground truth cos(x)')
plt.plot(x_dense, y_float, 'b-',  linewidth=1.5, label='Float model prediction')
plt.title('Float Model vs Ground Truth')
plt.xlabel('x (radians)'); plt.ylabel('cos(x)')
plt.legend(); plt.tight_layout()
plt.savefig('plot_float_prediction.png', dpi=150); plt.show()
print('Saved: plot_float_prediction.png')

## Step 4 — Save Float TFLite Model (unquantized baseline)

In [ ]:
converter_float = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float    = converter_float.convert()

with open('cosine_float.tflite', 'wb') as f:
    f.write(tflite_float)

float_size = len(tflite_float)
print(f'Float TFLite size : {float_size} bytes  ({float_size/1024:.1f} KB)')

## Step 5 — Full Integer (int8) Quantization

All four requirements from section 3.2 are met:
- `converter.optimizations = [tf.lite.Optimize.DEFAULT]`
- `representative_dataset` with 100 training samples
- `target_spec.supported_ops = [TFLITE_BUILTINS_INT8]`
- `inference_input_type = inference_output_type = tf.int8`

In [ ]:
def representative_dataset():
    """Yield 100 representative float32 samples from the training set."""
    for i in range(100):
        yield [np.array([[x_train[i]]], dtype=np.float32)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations              = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset     = representative_dataset
converter_int8.target_spec.supported_ops  = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type       = tf.int8
converter_int8.inference_output_type      = tf.int8

tflite_int8 = converter_int8.convert()

with open('cosine_int8.tflite', 'wb') as f:
    f.write(tflite_int8)

int8_size = len(tflite_int8)
print(f'Int8 TFLite size  : {int8_size} bytes  ({int8_size/1024:.1f} KB)')
print(f'Float TFLite size : {float_size} bytes  ({float_size/1024:.1f} KB)')
print(f'Size reduction    : {float_size/int8_size:.2f}x smaller')

## Step 6 — Evaluate the Int8 Model in Python

In [ ]:
# Load the int8 interpreter
interp = tf.lite.Interpreter(model_content=tflite_int8)
interp.allocate_tensors()

inp_det  = interp.get_input_details()[0]
out_det  = interp.get_output_details()[0]

in_scale,  in_zp  = inp_det['quantization']
out_scale, out_zp = out_det['quantization']

print(f'Input  quantization : scale={in_scale:.6f}, zero_point={in_zp}')
print(f'Output quantization : scale={out_scale:.6f}, zero_point={out_zp}')

# Run inference on the test set
y_int8 = []
for x_val_single in x_test:
    # Quantize float -> int8
    x_q = np.clip(
        np.round(x_val_single / in_scale + in_zp), -128, 127
    ).astype(np.int8)
    interp.set_tensor(inp_det['index'], np.array([[x_q]], dtype=np.int8))
    interp.invoke()
    # Dequantize int8 -> float
    y_q = interp.get_tensor(out_det['index'])[0][0]
    y_int8.append((float(y_q) - out_zp) * out_scale)

y_int8 = np.array(y_int8)
int8_mae = np.mean(np.abs(y_int8 - y_test))
print(f'\nInt8 model — Test MAE : {int8_mae:.6f}')

### 6a — Three-Curve Comparison Plot (Ground Truth / Float / Int8)

In [ ]:
# Run int8 on dense grid for smooth plot
y_int8_dense = []
for xv in x_dense:
    x_q = np.clip(np.round(xv / in_scale + in_zp), -128, 127).astype(np.int8)
    interp.set_tensor(inp_det['index'], np.array([[x_q]], dtype=np.int8))
    interp.invoke()
    y_q = interp.get_tensor(out_det['index'])[0][0]
    y_int8_dense.append((float(y_q) - out_zp) * out_scale)
y_int8_dense = np.array(y_int8_dense)

plt.figure(figsize=(9, 4))
plt.plot(x_dense, y_gt,         'k--', linewidth=1.5, label='Ground truth cos(x)')
plt.plot(x_dense, y_float,      'b-',  linewidth=1.5, label='Float32 prediction')
plt.plot(x_dense, y_int8_dense, 'r-',  linewidth=1.5, label='Int8 prediction')
plt.title('Ground Truth vs Float32 vs Int8 Predictions')
plt.xlabel('x (radians)'); plt.ylabel('cos(x)')
plt.legend(); plt.tight_layout()
plt.savefig('plot_three_curves.png', dpi=150); plt.show()
print('Saved: plot_three_curves.png')

## Step 7 — Generate `model.h` (C Byte Array)

Equivalent to `xxd -i cosine_int8.tflite > model.h`.  
The array is named `g_model[]` and marked `alignas(8)` as required by section 3.2 step 5.

In [ ]:
import binascii

hex_data = binascii.hexlify(tflite_int8).decode('ascii')
# Format as C hex literals, 12 per line for readability
bytes_list = ['0x' + hex_data[i:i+2] for i in range(0, len(hex_data), 2)]
lines      = []
for i in range(0, len(bytes_list), 12):
    lines.append('  ' + ', '.join(bytes_list[i:i+12]))
c_array = ',\n'.join(lines)

header = f"""#ifndef COSINE_MODEL_H
#define COSINE_MODEL_H

#include <stdint.h>

// Auto-generated from cosine_int8.tflite
// Model size: {int8_size} bytes
// Input  quantization : scale={in_scale:.8f}, zero_point={in_zp}
// Output quantization : scale={out_scale:.8f}, zero_point={out_zp}

alignas(8) const unsigned char g_model[] = {{
{c_array}
}};
const unsigned int g_model_len = {int8_size};

#endif  // COSINE_MODEL_H
"""

with open('model.h', 'w') as f:
    f.write(header)

print(f'model.h generated — {int8_size} bytes embedded')
print(f'Input  scale={in_scale:.8f}  zero_point={in_zp}')
print(f'Output scale={out_scale:.8f}  zero_point={out_zp}')

## Step 8 — Summary & Download Files

In [ ]:
print('=' * 55)
print('  ASSIGNMENT 3 — SUMMARY')
print('=' * 55)
print(f'  Samples              : {SAMPLES} (60/20/20 split)')
print(f'  Architecture         : Dense(16,relu) x2 -> Dense(1)')
print(f'  Trainable params     : {model.count_params()}')
print(f'  Float model MAE      : {test_mae:.6f}')
print(f'  Int8 model MAE       : {int8_mae:.6f}')
print(f'  Float TFLite size    : {float_size} bytes')
print(f'  Int8 TFLite size     : {int8_size} bytes')
print(f'  Size reduction       : {float_size/int8_size:.2f}x')
print(f'  Input  scale/zp      : {in_scale:.6f} / {in_zp}')
print(f'  Output scale/zp      : {out_scale:.6f} / {out_zp}')
print('=' * 55)
print('Files ready to download:')
print('  cosine_float.tflite')
print('  cosine_int8.tflite')
print('  model.h')
print('  plot_dataset.png')
print('  plot_loss_curves.png')
print('  plot_float_prediction.png')
print('  plot_three_curves.png')

In [ ]:
# ---- Download all files from Colab ----
try:
    from google.colab import files
    for fname in [
        'cosine_float.tflite',
        'cosine_int8.tflite',
        'model.h',
        'plot_dataset.png',
        'plot_loss_curves.png',
        'plot_float_prediction.png',
        'plot_three_curves.png',
    ]:
        files.download(fname)
        print(f'Downloading {fname}...')
except ImportError:
    print('Not running in Colab — files are saved in the current directory.')